# Bronze → Silver loader (PySpark + Delta MERGE)

The same load pattern as the T-SQL procedures (stage → update → insert → soft delete), written once
in PySpark and driven by a **config list**. Adding a new Silver table means adding one config entry,
not writing a new procedure.

- Reads Bronze tables from the lakehouse
- Casts and renames columns, keeps the newest row per key
- Builds a row hash, then runs a Delta `MERGE` into the Silver lakehouse table
- Soft-deletes keys that disappeared from Bronze (`whenNotMatchedBySource`)

**Status:** designed for the next phase of the project; not yet run in production.

In [ ]:
# Parameters cell (mark as "parameter cell" in Fabric so the pipeline can pass values)
run_id = "manual-run"
tables_to_load = "all"          # or a comma list, e.g. "supplier,transaction_source"
bronze_schema = "bronze"
silver_schema = "silver"

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# One entry per Silver table. columns = {silver_name: (bronze_column, spark_type)}
CONFIG = [
    {
        "name": "supplier",
        "bronze": "br_oracle_fusion_fscm_prcextractam_pozbiccextractam_supplier",
        "keys": ["vendor_id"],
        "columns": {
            "vendor_id":       ("VendorId", "bigint"),
            "party_id":        ("PartyId", "bigint"),
            "supplier_number": ("Segment1", "string"),
        },
        "last_update": "LastUpdateDate",
        "filter": None,
    },
    {
        "name": "transaction_source",
        "bronze": "br_oracle_fusion_fscm_prjextractam_pjfbiccextractam_transactionsource",
        "keys": ["transaction_source_id"],
        "columns": {
            "transaction_source_id":   ("TransactionSourceBasePEOTransactionSourceId", "bigint"),
            "user_transaction_source": ("TransactionSourceTLPEOUserTransactionSource", "string"),
        },
        "last_update": "TransactionSourceBasePEOLastUpdateDate",
        "filter": "TransactionSourceTLPEOLanguage = 'US'",
    },
    {
        "name": "expenditure_category",
        "bronze": "br_oracle_fusion_fscm_prjextractam_pjfbiccextractam_expenditurecategory",
        "keys": ["expenditure_category_id"],
        "columns": {
            "expenditure_category_id":   ("ExpenditureCategoryBasePEOExpenditureCategoryId", "bigint"),
            "expenditure_category_name": ("ExpenditureCategoryTLLangPEOName", "string"),
        },
        "last_update": "ExpenditureCategoryBasePEOLastUpdateDate",
        "filter": "ExpenditureCategoryTLLangPEOLanguage = 'US'",
    },
    {
        "name": "project_role",
        "bronze": "br_oracle_fusion_fscm_prjextractam_pjfbiccextractam_projectroletypetranslation",
        "keys": ["project_role_id"],
        "columns": {
            "project_role_id":   ("ProjectRoleTypeTranslationProjectRoleId", "bigint"),
            "project_role_name": ("ProjectRoleTypeTranslationProjectRoleName", "string"),
        },
        "last_update": "ProjectRoleTypeTranslationLastUpdateDate",
        "filter": "ProjectRoleTypeTranslationLanguage = 'US'",
    },
]

In [ ]:
def build_stage(cfg):
    """Bronze -> clean stage: filter, cast, rename, newest row per key, row hash."""
    df = spark.read.table(f"{bronze_schema}.{cfg['bronze']}")
    if cfg["filter"]:
        df = df.where(cfg["filter"])

    cols = [F.col(src).cast(typ).alias(name) for name, (src, typ) in cfg["columns"].items()]
    cols += [F.col(cfg["last_update"]).cast("timestamp").alias("source_last_update_date"),
             F.col("_LoadDTS").alias("_load_dts")]
    df = df.select(*cols)

    newest = Window.partitionBy(*cfg["keys"]).orderBy(F.col("source_last_update_date").desc_nulls_last(),
                                                       F.col("_load_dts").desc_nulls_last())
    df = df.withColumn("_rn", F.row_number().over(newest)).where("_rn = 1").drop("_rn", "_load_dts")

    hash_cols = [c for c in cfg["columns"] if c not in cfg["keys"]]
    df = df.withColumn("_row_hash",
                       F.sha2(F.concat_ws("|", *[F.coalesce(F.col(c).cast("string"), F.lit("")) for c in hash_cols]), 256))
    return df


def merge_into_silver(cfg, stage, run_id):
    """Delta MERGE: update changed rows, insert new keys, soft-delete keys gone from Bronze."""
    target_name = f"{silver_schema}.{cfg['name']}"
    now = F.current_timestamp()
    stage = (stage.withColumn("_silver_loaded_at", now)
                  .withColumn("_is_deleted", F.lit(False))
                  .withColumn("_run_id", F.lit(run_id)))

    if not spark.catalog.tableExists(target_name):
        stage.write.format("delta").saveAsTable(target_name)
        return

    target = DeltaTable.forName(spark, target_name)
    on = " AND ".join([f"t.{k} = s.{k}" for k in cfg["keys"]])
    data_cols = list(cfg["columns"]) + ["source_last_update_date", "_row_hash"]

    (target.alias("t")
        .merge(stage.alias("s"), on)
        .whenMatchedUpdate(
            condition="t._row_hash <> s._row_hash OR t._is_deleted = true",
            set={**{c: f"s.{c}" for c in data_cols},
                 "_silver_loaded_at": "current_timestamp()", "_is_deleted": "false",
                 "_run_id": f"'{run_id}'"})
        .whenNotMatchedInsertAll()
        .whenNotMatchedBySourceUpdate(
            condition="t._is_deleted = false",
            set={"_is_deleted": "true", "_silver_loaded_at": "current_timestamp()",
                 "_run_id": f"'{run_id}'"})
        .execute())

In [ ]:
def run_dq(cfg, run_id):
    """Same tests as the T-SQL DQ procedures. Returns a list of result rows."""
    t = spark.read.table(f"{silver_schema}.{cfg['name']}")
    active = t.where("_is_deleted = false")
    keys = cfg["keys"]
    dup = active.groupBy(*keys).count().where("count > 1").count()
    nulls = active.where(" OR ".join([f"{k} IS NULL" for k in keys])).count()
    bronze_keys = build_stage(cfg).select(*keys).distinct().count()
    return [
        (run_id, f"silver.{cfg['name']}", "key is unique", "ERROR", dup),
        (run_id, f"silver.{cfg['name']}", "key not null", "ERROR", nulls),
        (run_id, f"silver.{cfg['name']}", "row count matches bronze", "ERROR", abs(active.count() - bronze_keys)),
    ]


selected = CONFIG if tables_to_load == "all" else [c for c in CONFIG if c["name"] in tables_to_load.split(",")]
results = []
for cfg in selected:
    stage = build_stage(cfg)
    merge_into_silver(cfg, stage, run_id)
    results += run_dq(cfg, run_id)
    print(f"loaded {cfg['name']}")

dq = (spark.createDataFrame(results, "run_id string, table_name string, test_name string, severity string, failed_rows long")
           .withColumn("checked_at", F.current_timestamp()))
dq.write.format("delta").mode("append").saveAsTable("ops.dq_result")
display(dq)

In [ ]:
# Data quality gate: fail the notebook (and so the pipeline) if any ERROR test failed.
errors = dq.where("severity = 'ERROR' AND failed_rows > 0").count()
if errors > 0:
    raise Exception(f"DQ gate failed: {errors} ERROR test(s) with failed rows in run {run_id}")
mssparkutils.notebook.exit(f"OK: {len(selected)} tables loaded, 0 DQ errors")